# Pauli propagation of Q# circuits

This notebook computes expectation values of circuits written in Q# by Pauli propagation, with the `paulipropagation-qdk` package. An observable is evolved backwards through the circuit as a sum of Pauli strings, and the small terms of the sum are dropped on the way. No quantum state is held, so the method reaches circuits of a hundred qubits and more, as long as the observable stays a sum of not too many Pauli strings.

The packages `qdk` and `paulipropagation-qdk` must be installed.

In [ ]:
import math
import random
import time

from qdk import qsharp
from qdk.simulation import NoiseConfig

import paulipropagation_qdk as pp

## A circuit from Q#

The circuit is the evolution of a chain of spins with an interaction between neighbours and a transverse field, in `layers` steps. It is compiled to QIR for the base profile, and `circuit_from_qir` reads the gates of the QIR program.

In [ ]:
qsharp.init(target_profile=qsharp.TargetProfile.Base)
qsharp.eval("""
operation Evolve(n : Int, layers : Int, coupling : Double, field : Double) : Result[] {
    use qs = Qubit[n];
    for _ in 1..layers {
        for i in 0..n - 2 {
            Rzz(coupling, qs[i], qs[i + 1]);
        }
        for q in qs {
            Rx(field, q);
        }
    }
    MResetEachZ(qs)
}
""")


def evolution(n, layers, coupling=0.2, field=0.3, noise=None):
    qir = str(qsharp.compile(f"Evolve({n}, {layers}, {coupling}, {field})"))
    return pp.circuit_from_qir(qir, noise=noise)


circuit = evolution(12, 8)
circuit

## An expectation value

The observable is `Z` on one qubit in the middle of the chain. `expectation` propagates it through the circuit and returns its value in the state that the circuit prepares from all qubits in `|0>`.

In [ ]:
observable = pp.PauliSum(circuit.num_qubits)
observable.add("Z", [5])

exact = pp.expectation(circuit, observable, min_abs_coeff=0.0)
exact

Nothing was dropped here, so this is the exact value. On 12 qubits the QDK simulator can run the same program, and the average of the measurement results of qubit 5 over many shots must agree with it, up to the statistical error of the shots.

In [ ]:
shots = 20_000
results = qsharp.run("Evolve(12, 8, 0.2, 0.3)", shots)
sampled = sum(1 if shot[5] == qsharp.Result.Zero else -1 for shot in results) / shots
print(f"propagated {exact:.4f}   sampled {sampled:.4f} +- {1 / math.sqrt(shots):.4f}")

## A hundred qubits

The same circuit on 100 qubits is far beyond a state vector. The propagated observable grows too, and it is kept small by dropping terms: `max_weight` drops Pauli strings with more non-identity operators than the given number, and `min_abs_coeff` drops terms with small coefficients. The result is an approximation, and it converges as the truncation is loosened.

In [ ]:
circuit = evolution(100, 20)
observable = pp.PauliSum(100)
observable.add("Z", [50])

for max_weight in (2, 4, 6, 8, 10):
    start = time.perf_counter()
    evolved = pp.propagate(circuit, observable, max_weight=max_weight, min_abs_coeff=1e-7)
    seconds = time.perf_counter() - start
    print(f"max_weight {max_weight:2}   {len(evolved):8} terms   <Z> = {evolved.overlap_with_zero():.8f}   {seconds:.2f} s")

`propagate` returns the propagated observable, which holds more than one number: its overlap with any product state or with a state given as a Pauli sum is an expectation value in that state.

In [ ]:
evolved = pp.propagate(circuit, observable, max_weight=8, min_abs_coeff=1e-7)
print("all qubits |0>:", evolved.overlap_with_zero())
print("all qubits |+>:", evolved.overlap_with_plus())
print("qubit 50 flipped:", evolved.overlap_with_computational([50]))
print("largest terms:")
for label, coeff in sorted(evolved.terms(), key=lambda term: -abs(term[1]))[:3]:
    print("  ", label.strip("I"), "from qubit", len(label) - len(label.lstrip("I")), coeff)

## The error of a truncation

`estimate_mse` estimates the mean squared error that a truncation by weight makes, from random paths through the circuit and without propagating. The average is over the angles of the rotations, each drawn from `[-thetas, thetas]` whatever angle the circuit gives it, so it tells how the truncation does on circuits of this shape with angles up to that size, rather than on this one circuit.

The cell below compares the estimate with the error that the truncation actually makes on a few circuits with random angles, measured against a much looser truncation.

In [ ]:
radius = 0.6
weights = (1, 2, 3, 4)

rng = random.Random(1)
squared_errors = {max_weight: [] for max_weight in weights}
for _ in range(50):
    drawn = circuit.with_angles([rng.uniform(-radius, radius) for _ in circuit.angles])
    reference = pp.expectation(drawn, observable, max_weight=10, min_abs_coeff=1e-7)
    for max_weight in weights:
        value = pp.expectation(drawn, observable, max_weight=max_weight, min_abs_coeff=1e-7)
        squared_errors[max_weight].append((value - reference) ** 2)

for max_weight in weights:
    estimate = pp.estimate_mse(circuit, observable, 200_000, thetas=radius, max_weight=max_weight, seed=1)
    measured = sum(squared_errors[max_weight]) / len(squared_errors[max_weight])
    print(f"max_weight {max_weight}   estimated error {math.sqrt(estimate):.3f}   measured on 50 circuits {math.sqrt(measured):.3f}")

## Noise

With a `NoiseConfig` of the QDK, the Pauli errors of every gate become channels of the circuit. The propagation then gives the expectation value of the noisy circuit, which the simulators of the QDK would estimate from many shots. Noise damps the Pauli strings of high weight, so a noisy circuit is easier to propagate than a noiseless one.

In [ ]:
for strength in (0.0, 0.002, 0.01):
    noise = NoiseConfig()
    noise.rzz.set_depolarizing(strength)
    noise.rx.set_depolarizing(strength / 10)
    noisy = evolution(100, 20, noise=noise)
    evolved = pp.propagate(noisy, observable, max_weight=8, min_abs_coeff=1e-7)
    print(f"depolarizing {strength:5}   {len(evolved):8} terms   <Z> = {evolved.overlap_with_zero():.8f}")

## Gradients

`gradient` returns the expectation value together with its derivatives with respect to the angles of all rotations, of which a circuit of 12 layers has 2,388. The observable is propagated back through the circuit once, and then the circuit is undone gate by gate, which gives one derivative at every rotation. The parameter-shift rule would take two propagations for every angle.

With truncation the gradient is an approximation, like the value. Here it is checked on the largest derivative against a finite difference.

In [ ]:
circuit = evolution(100, 12)
options = dict(max_weight=8, min_abs_coeff=1e-7)

start = time.perf_counter()
value, derivatives = pp.gradient(circuit, observable, **options)
print(f"{len(derivatives)} derivatives in {time.perf_counter() - start:.2f} s, value {value:.8f}")

k = max(range(len(derivatives)), key=lambda i: abs(derivatives[i]))
step = 1e-4
angles = circuit.angles
shifted = []
for sign in (1, -1):
    moved = angles[:k] + [angles[k] + sign * step] + angles[k + 1:]
    shifted.append(pp.expectation(circuit.with_angles(moved), observable, **options))
print(f"derivative {k}: {derivatives[k]:.6f}, finite difference {(shifted[0] - shifted[1]) / (2 * step):.6f}")

## Many angles with a surrogate

Which Pauli strings the propagation reaches does not depend on the angles, and neither does a truncation by weight. A `Surrogate` propagates once for all angles and records how every coefficient is made of sines and cosines of them. Evaluating it for given angles gives the same sum as a propagation with those angles, without finding, ordering and merging the Pauli strings again. That pays when the same circuit is needed for many angles, as in a sweep or an optimization.

In [ ]:
n, layers = 40, 10
circuit = evolution(n, layers)
observable = pp.PauliSum(n)
observable.add("Z", [n // 2])

start = time.perf_counter()
surrogate = pp.Surrogate(circuit, observable, max_weight=5)
print(surrogate, f"built in {time.perf_counter() - start:.3f} s")

# The angles in the order of the circuit: in every layer the couplings, then the fields.
fields = [0.02 * step for step in range(51)]
sweep = [([0.2] * (n - 1) + [field] * n) * layers for field in fields]

start = time.perf_counter()
values = [surrogate.evaluate(angles).overlap_with_zero() for angles in sweep]
print(f"{len(sweep)} evaluations in {time.perf_counter() - start:.3f} s")

start = time.perf_counter()
propagated = [pp.expectation(circuit.with_angles(angles), observable, max_weight=5, min_abs_coeff=0.0) for angles in sweep]
print(f"{len(sweep)} propagations in {time.perf_counter() - start:.3f} s, the same values: {values == propagated}")

for field, value in list(zip(fields, values))[::10]:
    print(f"field {field:.1f}   <Z> = {value:.6f}")

## More

The README of the package describes the other features: states propagated forwards, gates and channels given as matrices, complex coefficients, merging of terms by symmetry, propagation with resampling, and the truncation options.